``` markdown
Node A
   │
   ▼
Node B
   │
   ▼
Node C
```
This is called sequential Excecution. only one node runs at a time. 
the excecution order is ```NodeA --> Node B --> Node C. very simple. only one node updates the state at a time. No conflicts

The concurrent means: Multiple tasks are executing during the same period of time instead of waiting for one another to finish. 

Conecurrent does not necessarily mean they execute on different CPUs or exactly at the same nanosecond.

The workflow allows multiple independent tasks to make progress without forcing a strict one-after-another order.

``` markdown
          Node A
         /
Start ──┤
         \
          Node B

```
Here nodes can run independently.

#### Why concurrency Exists in Graphs.

In a simple linear graph, nodes run one after another — A then B then C. There is no concurrency problem because only one node writes to state at a time.

But real agentic workflows often need parallel execution. For example:

* Search 3 different databases simultaneously
* Run a critic and a fact-checker at the same time
* Call 5 different APIs in parallel and collect all results


#### What a Concurrent Update Actually Is

When two nodes run in parallel and both return updates for the same state key, LangGraph must decide: what do I do with two values for the same key?
* If the key has no reducer, LangGraph has no rule to resolve the conflict — it raises an error.
* If the key has a reducer, LangGraph applies the reducer function to combine both values — no conflict, both contributions survive.

#### How LangGraph Creates Parallel Branches

You use add_conditional_edges with a special Send API, or more simply you add edges from one node to multiple nodes at once. LangGraph will fan out and execute all target nodes in parallel.

``` python
# Fan-out: node A triggers B and C simultaneously
builder.add_edge("node_a", "node_b")
builder.add_edge("node_a", "node_c")
# node_b and node_c run in parallel
# then both write to state at the same time
```



In [1]:
#### The Conflict Scenario — No Reducer

from typing import TypedDict
from langgraph.graph import StateGraph, END

class BadState(TypedDict):
    result: str    # plain field, no reducer

def node_a(state: BadState):
    return {"result": "answer from A"}

def node_b(state: BadState):
    return {"result": "answer from B"}

def entry_node(state: BadState):
    return {}   # just a pass-through to trigger parallel nodes

builder = StateGraph(BadState)
builder.add_node("entry",  entry_node)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)

builder.set_entry_point("entry")
builder.add_edge("entry", "node_a")   # both edges from same node
builder.add_edge("entry", "node_b")   # means parallel execution

builder.add_edge("node_a", END)
builder.add_edge("node_b", END)

graph = builder.compile()

# THIS WILL RAISE InvalidUpdateError
# because node_a and node_b both write to "result"
# and there is no reducer to resolve it
graph.invoke({"result": ""})

InvalidUpdateError: At key 'result': Can receive only one value per step. Use an Annotated key to handle multiple values.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_CONCURRENT_GRAPH_UPDATE

langgraph.errors.InvalidUpdateError:
At key 'result': Can't apply multiple writes to the same key without a reducer.


The Fix — Add a Reducer


In [2]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
import operator

class GoodState(TypedDict):
    results: Annotated[list, operator.add]   # reducer — both values survive

def node_a(state: GoodState):
    return {"results": ["answer from A"]}

def node_b(state: GoodState):
    return {"results": ["answer from B"]}

def entry_node(state: GoodState):
    return {}

builder = StateGraph(GoodState)
builder.add_node("entry",  entry_node)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)

builder.set_entry_point("entry")
builder.add_edge("entry", "node_a")
builder.add_edge("entry", "node_b")
builder.add_edge("node_a", END)
builder.add_edge("node_b", END)

graph = builder.compile()

final = graph.invoke({"results": []})
print(final)
# {"results": ["answer from A", "answer from B"]}
# Both values survived — no conflict

{'results': ['answer from A', 'answer from B']}


In [4]:
# Full Realistic Example — Parallel Research Agents
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
import operator
import time

# ─────────────────────────────────────────
# STATE
# ─────────────────────────────────────────

class ResearchState(TypedDict):
    query: str
    findings:  Annotated[list, operator.add]  # parallel nodes all write here
    errors:    Annotated[list, operator.add]  # parallel nodes all write here
    completed: Annotated[list, operator.add]  # tracks which agents finished


# ─────────────────────────────────────────
# ENTRY NODE
# ─────────────────────────────────────────

def coordinator(state: ResearchState):
    print(f"\n[coordinator] Starting research for: '{state['query']}'")
    print(f"[coordinator] Dispatching 3 parallel researcher agents...\n")
    return {}   # nothing to update, just triggers the fan-out


# ─────────────────────────────────────────
# PARALLEL RESEARCH NODES
# All 3 run at the same time
# All 3 write to the same state fields
# ─────────────────────────────────────────

def web_researcher(state: ResearchState):
    print(f"  [web_researcher]  running...")
    time.sleep(0.1)  # simulate I/O
    return {
        "findings":  [f"[web]  Found 5 articles about '{state['query']}'"],
        "completed": ["web_researcher"]
    }

def database_researcher(state: ResearchState):
    print(f"  [db_researcher]   running...")
    time.sleep(0.1)
    return {
        "findings":  [f"[db]   Found 12 records in internal database"],
        "completed": ["database_researcher"]
    }

def arxiv_researcher(state: ResearchState):
    print(f"  [arxiv_researcher] running...")
    time.sleep(0.1)
    return {
        "findings":  [f"[arxiv] Found 3 relevant papers"],
        "completed": ["arxiv_researcher"]
    }


# ─────────────────────────────────────────
# AGGREGATOR NODE — runs after all 3 finish
# ─────────────────────────────────────────

def aggregator(state: ResearchState):
    print(f"\n[aggregator] All agents finished.")
    print(f"[aggregator] Agents that completed : {state['completed']}")
    print(f"[aggregator] Total findings collected : {len(state['findings'])}")
    for f in state["findings"]:
        print(f"             - {f}")
    return {}


# ─────────────────────────────────────────
# GRAPH ASSEMBLY
# ─────────────────────────────────────────

builder = StateGraph(ResearchState)

builder.add_node("coordinator",         coordinator)
builder.add_node("web_researcher",      web_researcher)
builder.add_node("database_researcher", database_researcher)
builder.add_node("arxiv_researcher",    arxiv_researcher)
builder.add_node("aggregator",          aggregator)

# Entry
builder.set_entry_point("coordinator")

# Fan-out: coordinator triggers all 3 in parallel
builder.add_edge("coordinator", "web_researcher")
builder.add_edge("coordinator", "database_researcher")
builder.add_edge("coordinator", "arxiv_researcher")

# Fan-in: all 3 flow into aggregator
# LangGraph waits for ALL parallel nodes to finish before running aggregator
builder.add_edge("web_researcher",      "aggregator")
builder.add_edge("database_researcher", "aggregator")
builder.add_edge("arxiv_researcher",    "aggregator")

builder.add_edge("aggregator", END)

graph = builder.compile()


# ─────────────────────────────────────────
# INVOCATION
# ─────────────────────────────────────────

result = graph.invoke({
    "query":    "LangGraph concurrent state updates",
    "findings":  [],
    "errors":    [],
    "completed": []
})

print("\n" + "="*50)
print("FINAL STATE")
print("="*50)
print(f"findings  : {result['findings']}")
print(f"completed : {result['completed']}")



[coordinator] Starting research for: 'LangGraph concurrent state updates'
[coordinator] Dispatching 3 parallel researcher agents...

  [arxiv_researcher] running...
  [db_researcher]   running...
  [web_researcher]  running...

[aggregator] All agents finished.
[aggregator] Agents that completed : ['arxiv_researcher', 'database_researcher', 'web_researcher']
[aggregator] Total findings collected : 3
             - [arxiv] Found 3 relevant papers
             - [db]   Found 12 records in internal database
             - [web]  Found 5 articles about 'LangGraph concurrent state updates'

FINAL STATE
findings  : ['[arxiv] Found 3 relevant papers', '[db]   Found 12 records in internal database', "[web]  Found 5 articles about 'LangGraph concurrent state updates'"]
completed : ['arxiv_researcher', 'database_researcher', 'web_researcher']
